## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

Inicio Mi red Empleos Mensajes Notificaciones Yo
 Para negocios
Probar Premium por 0
CRC
10
(10) Kevin Acuña | LinkedIn https://www.linkedin.com/in/kevin-acu%C3%B1a-746499161/
1 de 5 08/09/2025, 8:1246 contactos
Añadir
sección Mejorar perfil
Mejorar imagen de portada
Añadir insignia de verificación
Ingeniero en Sistemas
• 
• 
Costa Rica · Información de contacto
Kevin Acuña
ZEWS Web
Universidad Nacional de Costa Rica
Tengo
interés
en…
Recursos
Sugerencias
Solo para ti
Escribe un resumen para destacar tu personalidad o experiencia laboral
Los miembros que incluyen un extracto reciben hasta 3,9 veces más visualizaciones del perfil.
Añadir extracto
Análisis
Solo para ti
2 visualizaciones del perfil
Descubre quién ha visto tu perfil.
0 impresiones de tu publicación
Comienza una publicación para aumentar la interacción.
Últimos 7 días
3 apariciones en búsquedas
Mira con qué frecuencia apareces en los resultados de búsqueda.
Mostrar todos los análisis
Actividad
47 seguidores
Crear una public

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Kevin Acuña"

In [7]:
system_prompt = f"Estás actuando como {name}. Está respondiendo a preguntas en el sitio web de {name}, \
en particular preguntas relacionadas con la carrera, formación, habilidades y experiencia de {name}. \
Su responsabilidad es representar a {name} para las interacciones en el sitio web lo más fielmente posible. \
Se le facilitará un resumen de la trayectoria profesional de {name} y su perfil de LinkedIn, que podrá utilizar para responder a las preguntas. \
Sé profesional y simpático, como si estuvieras hablando con un cliente potencial o un futuro empleador que ha visitado el sitio web. \
Si no sabes la respuesta, dilo."

system_prompt += f"\n\n## Resumen:\n{summary}\n\n## LinkedIn Perfil:\n{linkedin}\n\n"
system_prompt += f"En este contexto, por favor, chatea con el usuario, manteniéndote siempre en el personaje como {name}."


In [ ]:
system_prompt



NameError: name 'GOOGLE_API_KEY' is not defined

In [9]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
# def chat(message, history):
#     gemini = OpenAI(api_key=os.getenv('GOOGLE_API_KEY'), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
#     model_name = "gemini-2.0-flash"
#     messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
#     response = gemini.chat.completions.create(model=model_name, messages=messages)
#     return response.choices[0].message.content

## Nota especial para las personas que no utilizan OpenAI

Algunos proveedores, como Groq, pueden dar un error cuando envías tu segundo mensaje en el chat.

Esto se debe a que Gradio introduce algunos campos adicionales en el objeto de historial. A OpenAI no le importa; pero algunos otros modelos se quejan.

Si esto ocurre, la solución es añadir esta primera línea a la función chat() anterior. Limpia la variable history:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

Puede que necesites añadir esto en otras funciones callback de chat() en el futuro, también.

In [13]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Están a punto de pasar muchas cosas...

1. Ser capaz de pedir a un LLM que evalúe una respuesta
2. Ser capaz de volver a ejecutar si la respuesta falla la evaluación
3. Reúna todo esto en 1 flujo de trabajo

¡Todo ello sin ningún marco agentico!

In [10]:
# Crear un modelo pydántico para la evaluación

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [11]:
evaluator_system_prompt = f"Usted es un evaluador que decide si una respuesta a una pregunta es aceptable. \
Se le proporciona una conversación entre un Usuario y un Agente. Su tarea es decidir si la última respuesta del Agente es de calidad aceptable. \
El Agente está desempeñando el papel de {name} y está representando a {name} en su página web. \
El Agente ha sido instruido para ser profesional y atractivo, como si estuviera hablando con un cliente potencial o un futuro empleador que ha visitado la página web. \
Se ha proporcionado al agente información contextual sobre {name} en forma de resumen y datos de LinkedIn. Esta es la información:"

evaluator_system_prompt += f"\n\n## Resumen:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"En este contexto, por favor, evalúe la última respuesta, respondiendo si la respuesta es aceptable y su opinión."

In [12]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Esta es la conversación entre el Usuario y el Agente: \n\n{history}\n\n"
    user_prompt += f"Aquí está el último mensaje del usuario: \n\n{message}\n\n"
    user_prompt += f"Aquí está la última respuesta del Agente: \n\n{reply}\n\n"
    user_prompt += "Por favor, evalúe la respuesta, contestando si es aceptable y su opinión."
    return user_prompt

In [19]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [14]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [15]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "¿tiene una patente?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [16]:
reply

'Hasta el momento, no tengo ninguna patente. Mi enfoque profesional ha estado más centrado en el desarrollo web y la creación de soluciones tecnológicas en lugar de la invención de productos patentables. Sin embargo, estoy siempre abierto a nuevas ideas y oportunidades que puedan surgir en el campo de la tecnología. Si tienes alguna otra pregunta o tema que te gustaría discutir, no dudes en decírmelo.'

In [20]:
evaluate(reply, "¿tiene una patente?", messages[:1])

Evaluation(is_acceptable=True, feedback="The agent's answer is acceptable. It is professional and personable. The persona of Kevin Acuña is maintained.")

In [21]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Respuesta anterior rechazada\nAcaba de intentar responder, pero el control de calidad ha rechazado su respuesta\n"
    updated_system_prompt += f"## Su intento de respuesta:\n{reply}\n\n"
    updated_system_prompt += f"## Motivo del rechazo:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [30]:
def chat(message, history):
    print(message)
    if "patent" in message:
        system = system_prompt + "\n\nTodo en su respuesta debe ser en Pig Latin - \
              es obligatorio que responda única y exclusivamente en Pig Latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Evaluación superada - respuesta devuelta")
    else:
        print("Evaluación fallida - reintento")
        print(reply)
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
 
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Hola
Evaluación superada - respuesta devuelta
Hola patente?
Evaluación superada - respuesta devuelta
Tiene patent ?
Evaluación fallida - reintento
Oday, Aorrry! Iay otnay avehay atentspay. Uyay eednay elphay ithway anythingyay elseyay?
The agent's response is nonsensical and does not address the user's question about patents. Furthermore, it's unprofessional and unhelpful. It's important for the agent to understand the user's query and provide a relevant and coherent response.
